In [ ]:
!pip install -q scanpy leidenalg celltypist scvi-tools scikit-misc soupx-python

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
import leidenalg
import scipy.sparse as sp
import scvi
import celltypist
from pyscdblfinder import ScDblFinder
from pathlib import Path
import gc

In [ ]:
import subprocess
files_dir = '/covid_pbmc'

links = ['https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557327/suppl/GSM4557327%5F555%5F1%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557328/suppl/GSM4557328%5F555%5F2%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557329/suppl/GSM4557329%5F556%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557330/suppl/GSM4557330%5F557%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557331/suppl/GSM4557331%5F558%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557332/suppl/GSM4557332%5F559%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557333/suppl/GSM4557333%5F561%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557334/suppl/GSM4557334%5FHIP002%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557335/suppl/GSM4557335%5FHIP015%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557336/suppl/GSM4557336%5FHIP023%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557337/suppl/GSM4557337%5FHIP043%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557338/suppl/GSM4557338%5FHIP044%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz',
         'https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM4557nnn/GSM4557339/suppl/GSM4557339%5FHIP045%5Fcell%2Ecounts%2Ematrices%2Erds%2Egz']

#downloading the files using wget and decompressing them
for link in links:
  subprocess.run(['wget', '-P', files_dir, link], check = True)
  subprocess.run(f'gunzip {files_dir}/*.gz', shell = True, check = True)


In [ ]:
#Because the files are rds files i will use these libraries to convert to an anndata object
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects import r
from pathlib import Path

folder_path = Path('/covid_pbmc')
pandas2ri.activate()
readRDS = ro.r['readRDS']
adatas = {}

#dictionary for donor labels
donor_dict = {
              "GSM4557327": "covid_555_1",
              "GSM4557328": "covid_555_2",
              "GSM4557329": "covid_556",
              "GSM4557330": "covid_557",
              "GSM4557331": "covid_558",
              "GSM4557332": "covid_559",
              "GSM4557333": "covid_561",
              "GSM4557334": "HIP002",
              "GSM4557335": "HIP015",
              "GSM4557336": "HIP023",
              "GSM4557337": "HIP043",
              "GSM4557338": "HIP044",
              "GSM4557339": "HIP045"
}

for item in folder_path.iterdir():
  obj = readRDS(str(item.resolve())) #returns the path of the file as a string which readRDS expects
  exon = obj.rx2("exon")#the rds file contains 3 matrices exon, introns, and spanning, exon is the standard gene-expression matrix

  # convert R sparse matrix → SciPy sparse
  i = np.array(exon.slots["i"])
  p = np.array(exon.slots["p"])
  x = np.array(exon.slots["x"])
  dims = tuple(exon.slots["Dim"])

  X = sp.csc_matrix((x, i, p), shape=dims).T

  # build AnnData
  adata = ad.AnnData(X)
  adata.var_names = list(r['rownames'](exon))
  adata.obs_names = list(r['colnames'](exon))
  adata.var_names_make_unique()

  #extracts unique sample id and assigns it to a col
  sample_id = item.name.split('_')[0]
  adata.obs['sample'] = sample_id

  #extracts condition of the sample and assigns it to a col
  healthy_sample = item.name.split('_')[1]
  if healthy_sample.startswith('HIP'):
    adata.obs['condition'] = 'HEALTHY'
  else:
    adata.obs['condition'] = 'COVID'

  #adds a col for the donor label
  adata.obs['donor'] = adata.obs['sample'].map(donor_dict)

  #store the adata object back in the dictionary
  adatas[sample_id] = adata

In [ ]:
#ambient rna removal with soupx
from scipy.sparse import csr_matrix

for sample_id, adata in adatas.items():
    adata.var_names_make_unique()
    adata.layers['raw_data'] = adata.X.copy()

    temp = adata.copy()
    sc.pp.normalize_total(temp, target_sum=1e4)
    sc.pp.log1p(temp)
    sc.pp.pca(temp)
    sc.pp.neighbors(temp)
    sc.tl.leiden(temp, key_added='temp_leiden', flavor='igraph', directed=False, n_iterations=2)
    adata.obs['soupx_groups'] = temp.obs['temp_leiden']
    del temp

    soup = soupx.SoupChannel(tod=adata.X.T.tocsr()x,#since i dont have the raw data
                             toc=adata.X.T.tocsr(),#tod and toc are the same
                             )

    #using 20% based on a test i saw another lab run where 20 gave
    #the most ambient rna removal bc soupx is very conservative
    soup.set_contamination_fraction(0.2)

    soup.setClusters(adata.obs['soupx_groups'].astype(str).values)

    corrected_matrix = soupx.adjust_counts(soup, roundToInt=True)

    adata.layers['cleaned_counts'] = corrected_matrix.T.copy()
    adata.X = corrected_matrix.T.copy()

    adatas[sample_id] = adata
    del corrected_matrix
    gc.collect()

In [ ]:
#doublet removal
for sample_id, adata in adatas.items():
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=10)

    #calculates doublet rate at 1% per 1000 and caps it at 0.15
    #for adatas with more cells
    calculated_dbr = min((adata.n_obs/1000)*0.01, 0.15)

    doublet = ScDblFinder(adata, random_state=42)
    doublet.run(clusters='soupx_groups',
                dbr=calculated_dbr,
                dims=15,
                n_features=1000,
                artificial_doublets=3000,
                iter=2, nrounds=0.25,
                verbose=False
                )
    print(f'Doublets and singlets for {sample_id}: {adata.obs.scDblFinder_class.value_counts()}')
    adatas[sample_id] = adata

In [ ]:
#qc metrics
for sample_id, adata in adatas.items():
    adata.var['mt'] = adata.var.index.str.startswith('MT-')
    adata.var['rb'] = adata.var.index.str.startswith(("RPS", "RPL","FAU", "MRPL", "RSL"))
    adata.var['hb'] = adata.var_names.str.match('^HB[ABDEGMQZ]\d*$')

    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'rb', 'hb'], percent_top = [20], log1p=True, inplace=True)

    #visualizing the data
    p1 = sns.displot(adata.obs['total_counts'], bins = 100, kde = False)
    p2 = sc.pl.violin(adata, 'pct_counts_mt')
    p3 = sc.pl.scatter(adata, 'total_counts', 'n_genes_by_counts', color = 'pct_counts_mt')
    p4 = sc.pl.violin(adata, 'pct_counts_hb')

    adatas[sample_id] = adata

In [ ]:
from scipy.stats import median_abs_deviation

def is_outlier(adata, metric: str, nmads: int):
  M = adata.obs[metric]
  outlier = (M < np.median(M) - (nmads * median_abs_deviation(M))) | (M > np.median(M) + (nmads * median_abs_deviation(M)))
  return outlier

In [ ]:
#filtering the adata object
for sample_id, adata in adatas.items():
  adata.obs['outlier'] = (is_outlier(adata, 'log1p_total_counts', 5)
                          | is_outlier(adata, 'log1p_n_genes_by_counts', 5)
                          | is_outlier(adata, 'pct_counts_in_top_20_genes', 5))

  print(f'Total number of cells before filtering {sample_id}: {adata.X.shape}')
  print(f'Total number of cells before removing outliers from {sample_id}: {adata.X.shape}')
  adata = adata[(~adata.obs['outlier']),:]
  print(f'Total number of cells after removing outliers from {sample_id}: {adata.X.shape}')
  adata = adata[adata.obs['pct_counts_mt'] < 10,:]
  print(f'Total number of cells after removing high mt from {sample_id}: {adata.X.shape}')
  adata = adata[adata.obs['pct_counts_hb'] < 2,:]
  print(f'Total number of cells after removing high hb from {sample_id}: {adata.X.shape}')
  adata = adata[adata.obs['scDblFinder_class'] != 'doublet', :]
  print(f'Total number of cells after removing doublets from {sample_id}: {adata.X.shape}')
  print('  ')

  #visualizing the filtered data
  p1 = sns.displot(adata.obs['total_counts'], bins = 100, kde = False)
  p2 = sc.pl.violin(adata, 'pct_counts_mt')
  p3 = sc.pl.scatter(adata, 'total_counts', 'n_genes_by_counts', color = 'pct_counts_mt')
  p4 = sc.pl.violin(adata, 'pct_counts_hb')

  adata.layers['filtered'] = adata.X.copy()
  adata.write_h5ad(f'{sample_id}_filtered.h5ad', compression='gzip')
  adatas[sample_id] = adata

In [ ]:
#concatenating adatas
adata_all = sc.concat(adatas.values(), label='batch', keys=list(adatas.keys()), join='inner')
adata_all.obs_names_make_unique()

del adatas
gc.collect()

In [ ]:
#calculate hvg
sc.pp.highly_variable_genes(adata_all, n_top_genes = 2000, flavor = 'seurat_v3', layer = 'filtered', subset = False)

sc.pp.normalize_total(adata_all)
sc.pp.log1p(adata_all)

adata_hvg = adata_all[:,adata_all.var.highly_variable].copy()

In [ ]:
#visualizing samples before integrating
sc.pp.pca(adata_hvg, n_comps=30)
sc.pp.neighbors(adata_hvg, n_neighbors = 30, n_pcs = 30)
sc.tl.umap(adata_hvg)
sc.pl.umap(adata_hvg, color = ['batch', 'condition'], wspace = 0.5)

In [ ]:
#training with scvi
scvi.model.SCVI.setup_anndata(adata_hvg, layer = 'filtered', batch_key = 'batch', categorical_covariate_keys = ['donor'])
scvi_model = scvi.model.SCVI(adata_hvg, n_layers=2, n_latent=30)#2 layers allows the model to learn the complex relationship setting n_latent/ pcs to 30 to reduce risk of overfitting
scvi_model.train(accelerator="cpu", devices=1)
adata_hvg.obsm['X_scvi'] = scvi_model.get_latent_representation()

In [ ]:
#visualizing scvi results
sc.pp.neighbors(adata_hvg, use_rep = 'X_scvi')
sc.tl.leiden(adata_hvg, key_added = 'leiden_scvi', flavor = 'igraph', n_iterations = 2, resolution = 0.5)
sc.tl.umap(adata_hvg)
sc.pl.umap(adata_hvg, color = ['batch', 'condition'], wspace = 0.5)

In [ ]:
#creating general celltype labels with celltypist
temp_adata = adata_hvg.copy()
temp_adata.X = temp_adata.layers['filtered']

#reduces the memory of the temp matrix
temp_adata.X = temp_adata.X.astype('float32')
temp_adata.layers['filtered'] = temp_adata.layers['filtered'].astype('float32')
temp_adata.layers['raw_data'] = temp_adata.layers['raw_data'].astype('float32')
temp_adata.X = temp_adata.X.tocsr()

sc.pp.normalize_total(temp_adata, target_sum = 1e4)
sc.pp.log1p(temp_adata)

#using the high model bc i want broad labels
model = celltypist.models.Model.load("Immune_All_High.pkl")
prediction = celltypist.annotate(temp_adata, model = model, over_clustering='leiden_scvi')

adata_hvg.obs['rough_cell_type_label'] = prediction.predicted_labels
del temp_adata
gc.collect()

In [ ]:
sc.tl.umap(adata_hvg)
sc.pl.umap(adata_hvg, color = ['batch', 'rough_cell_type_label'], wspace = 0.5)

In [ ]:
#print # of unique labels
unique_labels = adata_hvg.obs['rough_cell_type_label'].nunique()
print(f"Total unique cell type labels found: {unique_labels}")

#view the full list of labels and their cell counts
print(adata_hvg.obs['rough_cell_type_label'].value_counts())

In [ ]:
#celltypist returned 22 labels 10 of these labels had < 20 cells
#i'm compressing it to less labels because that works better to scanvi integration
adata_hvg.obs['scanvi_input_labels'] = adata_hvg.obs['rough_cell_type_label'].astype(str)

#define a clean mapping to collapse the tiny tail groups (<50 cells)
refinement_map = {
    #keep the major clean populations exactly as they are
    'T cells': 'T cells',
    'Monocytes': 'Monocytes',
    'ILC': 'ILC',
    'B cells': 'B cells',
    'Plasma cells': 'Plasma cells',
    'DC': 'DC',
    'pDC': 'pDC',

    #merge minor myeloid/macrophage tiers into standard Monocytes
    'Macrophages': 'Monocytes',
    'Monocyte precursor': 'Monocytes',
    'Mono-mac': 'Monocytes',
    'Myelocytes': 'Monocytes',
    'Promyelocytes': 'Monocytes',

    #merge rare lymphoid precursors into T cells or B cells
    'Double-positive thymocytes': 'T cells',
    'Double-negative thymocytes': 'T cells',
    'DC precursor': 'DC',
    'B-cell lineage': 'B cells',

    #group active dividing cells cleanly
    'Cycling cells': 'Cycling cells',
    'Megakaryocytes/platelets': 'Megakaryocytes/platelets',
}

#apply the mapping
adata_hvg.obs['scanvi_input_labels'] = adata_hvg.obs['scanvi_input_labels'].map(refinement_map)

#turn completely foreign structural noise (Fibroblasts, Epithelial, HSC/MPP, Erythroid) into 'Unknown'
#scanvi will automatically classify these cells into their closest true immune group
adata_hvg.obs['scanvi_input_labels'] = adata_hvg.obs['scanvi_input_labels'].fillna('Unknown')

#22 groups reduced to 10
print('Finalized categories for SCANVI')
print(adata_hvg.obs['scanvi_input_labels'].value_counts())

In [ ]:
#scanvi integration
scanvi_model = scvi.model.SCANVI.from_scvi_model(scvi_model, labels_key = 'scanvi_input_labels', unlabeled_category = 'unknown')
scanvi_model.train(accelerator="cpu", devices=1, max_epochs=20)
adata_hvg.obsm['X_scanvi'] = scanvi_model.get_latent_representation()
adata_all.obsm['X_scanvi'] = scanvi_model.get_latent_representation()

In [ ]:
sc.pp.neighbors(adata_hvg, use_rep = 'X_scanvi')
sc.tl.umap(adata_hvg)
sc.pl.umap(adata_hvg, color = ['batch', 'condition', 'rough_cell_type_label'], wspace = 0.5)

In [ ]:
#saving copies of the files
adata_all.write_h5ad('adata_all_final.h5ad', compression='gzip')
adata_hvg.write_h5ad('adata_hvg_final.h5ad', compression='gzip')